# Face 01 · Dataset Preparation

**Training (identity classification):**
- **WebFace4M** (cleaned 4 M images / 200 k ids subset of WebFace260M) – best public quality-to-size ratio. Apply on the official site.
- **CASIA-WebFace** (0.5 M images / 10 k ids) – smaller, good for the first baseline.
- **DigiFace-1M** (synthetic, Microsoft) – adds pose / lighting variety without real people.

**Evaluation:** LFW, CFP-FP (profile), AgeDB-30 (age), **IJB-C** (TAR @ FAR 1e-4/1e-3), plus a consented
in-house set (Egyptian workers, garage lighting, masks / caps) for bias checks.

All of these are **research-licence** datasets → OK for the prototype; commercial launch = licensed data or SDK.
**No station worker faces are ever used for training** (spec section 11).

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Face_Recognition
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
FACES = DATA / "Faces"
TRAIN = FACES / "Train"          # ImageFolder layout: Train/<identity>/<img>.jpg  (aligned 112x112, notebook 02)
EVAL = FACES / "Eval"            # LFW / CFP-FP / AgeDB-30 pairs (.bin or folders + pairs.txt)
for d in (TRAIN, EVAL):
    d.mkdir(parents=True, exist_ok=True)

ids = [p for p in TRAIN.iterdir() if p.is_dir()]
print(len(ids), "identities")

In [ ]:
# Basic hygiene: drop identities with < 5 images, check duplicates by hash
import hashlib, collections
counts = {p.name: len(list(p.glob("*.jpg"))) for p in ids}
small = [k for k, v in counts.items() if v < 5]
print("identities < 5 images:", len(small))